In [30]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold , cross_val_score
from sklearn.linear_model import LinearRegression , Ridge , Lasso
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder , StandardScaler , OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor, AdaBoostRegressor
from xgboost import XGBRegressor
from sklearn.neural_network import MLPRegressor


from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

from sklearn.decomposition import PCA

In [31]:
df = pd.read_csv("/content/gurgaon_properties_post_feature_selection_v2.csv")

In [32]:
df.head()

,property_type,sector,price,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,luxury_category,floor_category
0,flat,sector 36,0.82,3.0,2.0,2,New Property,850.0,0.0,0.0,0.0,Low,Low Floor
1,flat,sector 89,0.95,2.0,2.0,2,New Property,1226.0,1.0,0.0,0.0,Low,Mid Floor
2,flat,sohna road,0.32,2.0,2.0,1,New Property,1000.0,0.0,0.0,0.0,Low,High Floor
3,flat,sector 92,1.60,3.0,4.0,3+,Relatively New,1615.0,1.0,0.0,1.0,High,Mid Floor
4,flat,sector 102,0.48,2.0,2.0,1,Relatively New,582.0,0.0,1.0,0.0,High,Mid Floor


In [33]:
# why we are using old dataset without encoding columns , because we want to do preprocessing , model training inside a pipeline.

In [34]:
df['furnishing_type'].value_counts()

,count
furnishing_type,
0.0,2349
1.0,1018
2.0,187


In [35]:
df['furnishing_type'] = df['furnishing_type'].replace({0.0:"unfurnished" , 1.0:"semifurnished",2.0:"furnished"})

In [36]:
df.head()

,property_type,sector,price,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,luxury_category,floor_category
0,flat,sector 36,0.82,3.0,2.0,2,New Property,850.0,0.0,0.0,unfurnished,Low,Low Floor
1,flat,sector 89,0.95,2.0,2.0,2,New Property,1226.0,1.0,0.0,unfurnished,Low,Mid Floor
2,flat,sohna road,0.32,2.0,2.0,1,New Property,1000.0,0.0,0.0,unfurnished,Low,High Floor
3,flat,sector 92,1.60,3.0,4.0,3+,Relatively New,1615.0,1.0,0.0,semifurnished,High,Mid Floor
4,flat,sector 102,0.48,2.0,2.0,1,Relatively New,582.0,0.0,1.0,unfurnished,High,Mid Floor


In [37]:
X = df.drop(columns=['price'])
y = df['price']

In [38]:
# Applying log1p transformation to target variable
y_transformed = np.log1p(y)

# Ordinal Encoding

In [39]:
columns_to_encode = ['property_type', 'sector', 'balcony',   'agePossession', 'furnishing_type', 'luxury_category', 'floor_category']

In [40]:
# Creating a Columns transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers = [
        ('num' , StandardScaler() , ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat' , OrdinalEncoder() , columns_to_encode)
    ],
    remainder = 'passthrough'
)

In [41]:
# creating a pipeline
pipeline = Pipeline(
    [
        ('preprocessor' , preprocessor),
        ('regressor' , LinearRegression())
    ]
)

In [42]:
# k-fold cross validation
kfold = KFold(n_splits = 10 , shuffle = True , random_state = 42)
scores = cross_val_score(pipeline , X , y_transformed , cv = kfold , scoring='r2')

In [43]:
scores.mean() , scores.std()

(np.float64(0.7363096633436828), np.float64(0.03238005754429932))

In [44]:
X_train , X_test , y_train , y_test = train_test_split(X, y_transformed, test_size=0.2, random_state=42)

In [45]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num', StandardScaler(),
                                                  ['bedRoom', 'bathroom',
                                                   'built_up_area',
                                                   'servant room',
                                                   'store room']),
                                                 ('cat', OrdinalEncoder(),
                                                  ['property_type', 'sector',
                                                   'balcony', 'agePossession',
                                                   'furnishing_type',
                                                   'luxury_category',
                                                   'floor_category'])])),
                ('regressor', LinearRegression())])

In [46]:
y_pred = pipeline.predict(X_test)

In [47]:
y_pred = np.expm1(y_pred)

In [48]:
mean_absolute_error(np.expm1(y_test), y_pred)

0.946382216008936

In [49]:
def scorer(model_name, model):

    output = []

    output.append(model_name)

    pipeline = Pipeline([
        ('preprocessor' , preprocessor),
        ('regressor',model)
    ])

    # kfold cross val score
    kfold = KFold(n_splits = 10 , shuffle = True , random_state = 42)
    scores = cross_val_score(pipeline , X , y_transformed , cv = kfold , scoring='r2')

    output.append(scores.mean())

    X_train , X_test , y_train , y_test = train_test_split(X, y_transformed, test_size=0.2, random_state=42)

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    y_pred = np.expm1(y_pred)

    output.append(mean_absolute_error(np.expm1(y_test) , y_pred))

    return output

In [51]:
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'rigde':Ridge(),
    'LASSO':Lasso(),
    'decision tree':DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees':ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost':AdaBoostRegressor(),
    'mlp':MLPRegressor(),
    'xgboost':XGBRegressor()
    }

In [53]:
model_output = []
for model_name, model in model_dict.items():
    model_output.append(scorer(model_name, model))

In [54]:
model_output

[['linear_reg', np.float64(0.7363096633436828), 0.946382216008936],
 ['svr', np.float64(0.7642021216646014), 0.8472636473483917],
 ['rigde', np.float64(0.7363125343993554), 0.9463387741853388],
 ['LASSO', np.float64(0.05943378064493573), 1.528905986892753],
 ['decision tree', np.float64(0.7726019523204382), 0.7629447376107996],
 ['random forest', np.float64(0.8819126370940644), 0.5362455363509996],
 ['extra trees', np.float64(0.867355194488493), 0.5482373692632695],
 ['gradient boosting', np.float64(0.8725996474148611), 0.575836417894087],
 ['adaboost', np.float64(0.7536173376782843), 0.8253669537373893],
 ['mlp', np.float64(0.8078967737268845), 0.6903223560970672],
 ['xgboost', np.float64(0.8894876835260124), 0.5040475127230885]]

In [56]:
model_df = pd.DataFrame(model_output , columns=['name', 'r2', 'mae'])
model_df.sort_values('mae')

,name,r2,mae
10,xgboost,0.889488,0.504048
5,random forest,0.881913,0.536246
6,extra trees,0.867355,0.548237
7,gradient boosting,0.872600,0.575836
9,mlp,0.807897,0.690322
4,decision tree,0.772602,0.762945
8,adaboost,0.753617,0.825367
1,svr,0.764202,0.847264
2,rigde,0.736313,0.946339
0,linear_reg,0.736310,0.946382


# OneHotEncoding

In [61]:
# Creating a Columns transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers = [
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat', OrdinalEncoder(), columns_to_encode),
        ('cat1', OneHotEncoder(drop='first'), ['sector', 'agePossession', 'furnishing_type'])
    ],
    remainder='passthrough'
)

In [62]:
# pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [63]:
# kfold cross val score
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

In [64]:
scores.mean(),scores.std()

(np.float64(0.8546112792716141), np.float64(0.01599323234861558))

In [65]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num', StandardScaler(),
                                                  ['bedRoom', 'bathroom',
                                                   'built_up_area',
                                                   'servant room',
                                                   'store room']),
                                                 ('cat', OrdinalEncoder(),
                                                  ['property_type', 'sector',
                                                   'balcony', 'agePossession',
                                                   'furnishing_type',
                                                   'luxury_category',
                                                   'floor_category']),
                                                 ('cat1',
                                                  OneHotEncoder(drop='first'),
                                                  ['sector', 'agePossession',
                                                   'furnishing_type'])])),
                ('regressor', LinearRegression())])

In [66]:
y_pred = pipeline.predict(X_test)

In [67]:
y_pred = np.expm1(y_pred)

In [68]:
mean_absolute_error(np.expm1(y_test), y_pred)

0.6497696680344619

In [69]:
def scorer(model_name, model):

    output = []

    output.append(model_name)

    pipeline = Pipeline([
        ('preprocessor' , preprocessor),
        ('regressor',model)
    ])

    # kfold cross val score
    kfold = KFold(n_splits = 10 , shuffle = True , random_state = 42)
    scores = cross_val_score(pipeline , X , y_transformed , cv = kfold , scoring='r2')

    output.append(scores.mean())

    X_train , X_test , y_train , y_test = train_test_split(X, y_transformed, test_size=0.2, random_state=42)

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    y_pred = np.expm1(y_pred)

    output.append(mean_absolute_error(np.expm1(y_test) , y_pred))

    return output

In [70]:
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'rigde':Ridge(),
    'LASSO':Lasso(),
    'decision tree':DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees':ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost':AdaBoostRegressor(),
    'mlp':MLPRegressor(),
    'xgboost':XGBRegressor()
    }

In [71]:
model_output = []
for model_name, model in model_dict.items():
    model_output.append(scorer(model_name, model))

/usr/local/lib/python3.11/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


In [72]:
model_output

[['linear_reg', np.float64(0.8546112792716141), 0.6497696680344619],
 ['svr', np.float64(0.7697515055540964), 0.8341243500492133],
 ['rigde', np.float64(0.8546761958777696), 0.6528987877965269],
 ['LASSO', np.float64(0.05943378064493573), 1.528905986892753],
 ['decision tree', np.float64(0.8054119121265801), 0.6720847435111061],
 ['random forest', np.float64(0.8913647229939059), 0.4988161178473235],
 ['extra trees', np.float64(0.8938640909877712), 0.47240712751425573],
 ['gradient boosting', np.float64(0.8766893469175943), 0.5693110403615981],
 ['adaboost', np.float64(0.7488735875357442), 0.8435158891666868],
 ['mlp', np.float64(0.8700765530905088), 0.539620163623174],
 ['xgboost', np.float64(0.8958499681743852), 0.493456263606726]]

In [73]:
model_df = pd.DataFrame(model_output, columns=['name', 'r2', 'mae'])
model_df.sort_values(['mae'])

,name,r2,mae
6,extra trees,0.893864,0.472407
10,xgboost,0.895850,0.493456
5,random forest,0.891365,0.498816
9,mlp,0.870077,0.539620
7,gradient boosting,0.876689,0.569311
0,linear_reg,0.854611,0.649770
2,rigde,0.854676,0.652899
4,decision tree,0.805412,0.672085
1,svr,0.769752,0.834124
8,adaboost,0.748874,0.843516


# OneHotEncoding with PCA

In [75]:
# Creating a Columns transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers = [
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat', OrdinalEncoder(), columns_to_encode),
        ('cat1', OneHotEncoder(drop='first', sparse_output=False), ['sector', 'agePossession'])
    ],
    remainder='passthrough'
)

In [76]:
# pipeline
pipeline = Pipeline([
      ('preprocessor', preprocessor),
      ('PCA', PCA(n_components=.95)),
      ('regressor', LinearRegression())
])

In [78]:
# kfold cross val score
kfold = KFold(n_splits = 10 , shuffle = True , random_state = 42)
scores = cross_val_score(pipeline , X , y_transformed , cv = kfold , scoring='r2')

In [79]:
scores.mean() , scores.std()

(np.float64(0.06225201431451134), np.float64(0.019860594071640165))

In [87]:
def scorer(model_name, model):

    output = []

    output.append(model_name)

    pipeline = Pipeline([
        ('preprocessor' , preprocessor),
        ('PCA',PCA(n_components=.95)),
        ('regressor',model)
    ])

    # kfold cross val score
    kfold = KFold(n_splits = 10 , shuffle = True , random_state = 42)
    scores = cross_val_score(pipeline , X , y_transformed , cv = kfold , scoring='r2')

    output.append(scores.mean())

    X_train , X_test , y_train , y_test = train_test_split(X, y_transformed, test_size=0.2, random_state=42)

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    y_pred = np.expm1(y_pred)

    output.append(mean_absolute_error(np.expm1(y_test) , y_pred))

    return output

In [88]:
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'rigde':Ridge(),
    'LASSO':Lasso(),
    'decision tree':DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees':ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost':AdaBoostRegressor(),
    'mlp':MLPRegressor(),
    'xgboost':XGBRegressor()
    }

In [89]:
model_output = []
for model_name, model in model_dict.items():
    model_output.append(scorer(model_name, model))

In [90]:
model_output

[['linear_reg', np.float64(0.06225201431451134), 1.5267074088549337],
 ['svr', np.float64(0.2180726480297686), 1.361198029862442],
 ['rigde', np.float64(0.062252015161791484), 1.5267074078044667],
 ['LASSO', np.float64(0.05967578446737004), 1.5287392557835464],
 ['decision tree', np.float64(0.6964420082698518), 0.761508966234373],
 ['random forest', np.float64(0.7624596204476547), 0.6567608565452452],
 ['extra trees', np.float64(0.7397910515580745), 0.704958207556113],
 ['gradient boosting', np.float64(0.6106227078866426), 0.9879063301936339],
 ['adaboost', np.float64(0.3087446390545809), 1.3865886908979623],
 ['mlp', np.float64(0.20976955134021943), 1.3926979378308397],
 ['xgboost', np.float64(0.6222047517390725), 0.9675805142651799]]

In [91]:
model_df = pd.DataFrame(model_output, columns=['name', 'r2', 'mae'])

In [92]:
model_df.sort_values(['mae'])

,name,r2,mae
5,random forest,0.762460,0.656761
6,extra trees,0.739791,0.704958
4,decision tree,0.696442,0.761509
10,xgboost,0.622205,0.967581
7,gradient boosting,0.610623,0.987906
1,svr,0.218073,1.361198
8,adaboost,0.308745,1.386589
9,mlp,0.209770,1.392698
2,rigde,0.062252,1.526707
0,linear_reg,0.062252,1.526707


# Target Encoding

In [94]:
!pip install category_encoders

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.7/85.7 kB 2.4 MB/s eta 0:00:00


In [96]:
import category_encoders as ce

# Creating a Columns transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers = [
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat', OrdinalEncoder(), columns_to_encode),
        ('cat1', OneHotEncoder(drop='first', sparse_output=False), ['sector', 'agePossession']),
        ('target_enc', ce.TargetEncoder(), ['sector'])
    ],
    remainder='passthrough'
)

In [97]:
# pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [98]:
# kfold cross val score
kfold = KFold(n_splits = 10 , shuffle = True , random_state = 42)
scores = cross_val_score(pipeline , X , y_transformed , cv = kfold , scoring='r2')

In [101]:
scores.mean(), scores.std()

(np.float64(0.854732932359442), np.float64(0.015972755156134895))

In [102]:
def scorer(model_name, model):

    output = []

    output.append(model_name)

    pipeline = Pipeline([
        ('preprocessor' , preprocessor),
        ('regressor',model)
    ])

    # kfold cross val score
    kfold = KFold(n_splits = 10 , shuffle = True , random_state = 42)
    scores = cross_val_score(pipeline , X , y_transformed , cv = kfold , scoring='r2')

    output.append(scores.mean())

    X_train , X_test , y_train , y_test = train_test_split(X, y_transformed, test_size=0.2, random_state=42)

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    y_pred = np.expm1(y_pred)

    output.append(mean_absolute_error(np.expm1(y_test) , y_pred))

    return output

In [103]:
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'rigde':Ridge(),
    'LASSO':Lasso(),
    'decision tree':DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees':ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost':AdaBoostRegressor(),
    'mlp':MLPRegressor(),
    'xgboost':XGBRegressor()
    }

In [104]:
model_output = []
for model_name, model in model_dict.items():
    model_output.append(scorer(model_name, model))

/usr/local/lib/python3.11/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


In [105]:
model_output

[['linear_reg', np.float64(0.854732932359442), 0.6491718597838219],
 ['svr', np.float64(0.7850445966497873), 0.8102763633996432],
 ['rigde', np.float64(0.8552252894671396), 0.6483482112400846],
 ['LASSO', np.float64(0.05943378064493573), 1.528905986892753],
 ['decision tree', np.float64(0.8390866179061243), 0.5656632108862523],
 ['random forest', np.float64(0.9021047225354891), 0.4568983109721027],
 ['extra trees', np.float64(0.9060376608904551), 0.43860475590829895],
 ['gradient boosting', np.float64(0.88961034871755), 0.5099623008835852],
 ['adaboost', np.float64(0.8128496871487967), 0.7165172341730054],
 ['mlp', np.float64(0.8745372158367705), 0.5260356103139927],
 ['xgboost', np.float64(0.9040029328338891), 0.46906252173850976]]

In [106]:
model_df = pd.DataFrame(model_output, columns=['name', 'r2', 'mae'])

In [107]:
model_df.sort_values(['mae'])

,name,r2,mae
6,extra trees,0.906038,0.438605
5,random forest,0.902105,0.456898
10,xgboost,0.904003,0.469063
7,gradient boosting,0.889610,0.509962
9,mlp,0.874537,0.526036
4,decision tree,0.839087,0.565663
2,rigde,0.855225,0.648348
0,linear_reg,0.854733,0.649172
8,adaboost,0.812850,0.716517
1,svr,0.785045,0.810276


In [108]:
# problem with target encoding is data leakage , because you are using avg_of_target columns , so before using target encoder make
# sure to split your data into train and test , so that test data is not used for target encoding , we are cross-val which handles it enternally.

# HyperParameter Tuning

In [124]:
from sklearn.model_selection import GridSearchCV

In [139]:
# Creating a Columns transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers = [
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat', OrdinalEncoder(), columns_to_encode),
        ('cat1', OneHotEncoder(drop='first', sparse_output=False), ['sector', 'agePossession']),
        ('target_enc', ce.TargetEncoder(), ['sector'])
    ],
    remainder='passthrough'
)

In [140]:
def scorer_tuning(model_name, model, param_grid):
    output = []

    output.append(model_name)

    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])

    kfold = KFold(n_splits=10, shuffle=True, random_state=42)

    search = GridSearchCV(pipeline, param_grid, cv=kfold, scoring='r2', n_jobs=-1, verbose=4)



    search.fit(X, y_transformed)

    output.append(search.best_params_)
    output.append(search.best_score_)

    pipe = search.best_estimator_

    X_train , X_test , y_train , y_test = train_test_split(X, y_transformed, test_size=0.2, random_state=42)

    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_pred = np.expm1(y_pred)

    output.append(mean_absolute_error(np.expm1(y_test) , y_pred))

    return output

In [141]:
model_output = []

In [142]:
# random forest
param_grid1 = {
    'regressor__n_estimators': [200, 500],
    'regressor__max_depth': [None, 30],
    'regressor__max_samples': [0.75, 1.0],
    'regressor__max_features': ['sqrt'],
    'regressor__min_samples_split': [2, 10],
    'regressor__min_samples_leaf': [1, 4],
    'regressor__bootstrap': [True]
}

model_name = 'Random Forest'
model = RandomForestRegressor()
model_output.append(scorer_tuning(model_name, model, param_grid1))

Fitting 10 folds for each of 32 candidates, totalling 320 fits


In [143]:
model_output

[['Random Forest',
  {'regressor__bootstrap': True,
   'regressor__max_depth': 30,
   'regressor__max_features': 'sqrt',
   'regressor__max_samples': 1.0,
   'regressor__min_samples_leaf': 1,
   'regressor__min_samples_split': 2,
   'regressor__n_estimators': 500},
  np.float64(0.8972196668880634),
  0.46750442216301413]]

In [144]:
# XGboost
param_grid2 = {
    'regressor__n_estimators': [200, 300],
    'regressor__max_depth': [5, 7],
    'regressor__learning_rate': [0.01, 0.1],
    'regressor__subsample': [0.9, 1.0],
    'regressor__colsample_bytree': [0.9, 1.0],
    'regressor__gamma': [0, 0.1],
    'regressor__reg_alpha': [0, 0.5],
    'regressor__reg_lambda': [0, 0.5]
}
model_name = 'XGBoost'
model = XGBRegressor()
model_output.append(scorer_tuning(model_name, model, param_grid2))

Fitting 10 folds for each of 256 candidates, totalling 2560 fits


In [145]:
model_output

[['Random Forest',
  {'regressor__bootstrap': True,
   'regressor__max_depth': 30,
   'regressor__max_features': 'sqrt',
   'regressor__max_samples': 1.0,
   'regressor__min_samples_leaf': 1,
   'regressor__min_samples_split': 2,
   'regressor__n_estimators': 500},
  np.float64(0.8972196668880634),
  0.46750442216301413],
 ['XGBoost',
  {'regressor__colsample_bytree': 0.9,
   'regressor__gamma': 0,
   'regressor__learning_rate': 0.1,
   'regressor__max_depth': 5,
   'regressor__n_estimators': 300,
   'regressor__reg_alpha': 0.5,
   'regressor__reg_lambda': 0.5,
   'regressor__subsample': 0.9},
  np.float64(0.9076500289770124),
  0.4801750471588596]]

In [147]:
# extra trees
param_grid3 = {
    'regressor__n_estimators': [200, 500],
    'regressor__max_depth': [None, 20],
    'regressor__min_samples_split': [2, 10],
    'regressor__min_samples_leaf': [1, 4],
    'regressor__max_features': ['sqrt', None],
    'regressor__bootstrap': [True, False]
}
model_name = 'Extra Trees'
model = ExtraTreesRegressor()
model_output.append(scorer_tuning(model_name, model, param_grid3))

Fitting 10 folds for each of 64 candidates, totalling 640 fits


In [148]:
model_output

[['Random Forest',
  {'regressor__bootstrap': True,
   'regressor__max_depth': 30,
   'regressor__max_features': 'sqrt',
   'regressor__max_samples': 1.0,
   'regressor__min_samples_leaf': 1,
   'regressor__min_samples_split': 2,
   'regressor__n_estimators': 500},
  np.float64(0.8972196668880634),
  0.46750442216301413],
 ['XGBoost',
  {'regressor__colsample_bytree': 0.9,
   'regressor__gamma': 0,
   'regressor__learning_rate': 0.1,
   'regressor__max_depth': 5,
   'regressor__n_estimators': 300,
   'regressor__reg_alpha': 0.5,
   'regressor__reg_lambda': 0.5,
   'regressor__subsample': 0.9},
  np.float64(0.9076500289770124),
  0.4801750471588596],
 ['Extra Trees',
  {'regressor__bootstrap': True,
   'regressor__max_depth': None,
   'regressor__max_features': None,
   'regressor__min_samples_leaf': 1,
   'regressor__min_samples_split': 2,
   'regressor__n_estimators': 500},
  np.float64(0.9084433419612274),
  0.4394608514155618]]

In [149]:
model_df = pd.DataFrame(model_output, columns=['name', 'best_params', 'best_score', 'mae'])

In [150]:
model_df.sort_values(['mae'])

,name,best_params,best_score,mae
2,Extra Trees,"{'regressor__bootstrap': True, 'regressor__max...",0.908443,0.439461
0,Random Forest,"{'regressor__bootstrap': True, 'regressor__max...",0.897220,0.467504
1,XGBoost,"{'regressor__colsample_bytree': 0.9, 'regresso...",0.907650,0.480175


# Exporting the model

In [151]:
preprocessor = ColumnTransformer(
    transformers = [
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat', OrdinalEncoder(), columns_to_encode),
        ('cat1', OneHotEncoder(drop='first', sparse_output=False), ['sector', 'agePossession']),
        ('target_enc', ce.TargetEncoder(), ['sector'])
    ],
    remainder='passthrough'
)

In [152]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', ExtraTreesRegressor(n_estimators=500, max_depth=None, max_features=None, min_samples_leaf=1, min_samples_split=2, bootstrap=True))
])

In [153]:
pipeline.fit(X, y_transformed)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num', StandardScaler(),
                                                  ['bedRoom', 'bathroom',
                                                   'built_up_area',
                                                   'servant room',
                                                   'store room']),
                                                 ('cat', OrdinalEncoder(),
                                                  ['property_type', 'sector',
                                                   'balcony', 'agePossession',
                                                   'furnishing_type',
                                                   'luxury_category',
                                                   'floor_category']),
                                                 ('cat1',
                                                  OneHotEncoder(drop='first',
                                                                sparse_output=False),
                                                  ['sector', 'agePossession']),
                                                 ('target_enc', TargetEncoder(),
                                                  ['sector'])])),
                ('regressor',
                 ExtraTreesRegressor(bootstrap=True, max_features=None,
                                     n_estimators=500))])

In [154]:
kfold = KFold(n_splits = 10 , shuffle = True , random_state = 42)
scores = cross_val_score(pipeline , X , y_transformed , cv = kfold , scoring='r2')
scores.mean(),scores.std()

(np.float64(0.908139175447328), np.float64(0.012616401539866403))

In [155]:
import pickle

with open('pipeline.pkl', 'wb') as file:
    pickle.dump(pipeline, file)

In [156]:
with open('df.pkl', 'wb') as file:
    pickle.dump(X, file)

# Trying out predictions

In [174]:
data = [['house', 'sector 49', 2, 2, '3+', 'New Property', 1150, 0, 0, 'unfurnished', 'Low', 'Low Floor']]
columns = ['property_type', 'sector', 'bedRoom', 'bathroom', 'balcony',
       'agePossession', 'built_up_area', 'servant room', 'store room',
       'furnishing_type', 'luxury_category', 'floor_category']

# Convert to DataFrame
one_df = pd.DataFrame(data, columns=columns)

one_df

,property_type,sector,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,luxury_category,floor_category
0,house,sector 49,2,2,3+,New Property,1150,0,0,unfurnished,Low,Low Floor


In [175]:
np.expm1(pipeline.predict(one_df))

array([2.09878431])